# ETL‑script

### Importeer modules, maak databaseverbinding en definieer hulpfuncties

In [60]:
import sqlite3
import pandas as pd
import datetime as dt
from pathlib import Path
from typing import List, Dict

In [61]:
# Verbind met SQLite en zet foreign keys aan
def connect_db(db_path: Path) -> sqlite3.Connection:
    con = sqlite3.connect(db_path)
    con.execute("PRAGMA foreign_keys = ON;")
    return con

In [62]:
# Maak of actualiseer de doelschema's.  Dim_Product en Dim_Klant krijgen SCD2‑kolommen
def prepare_target_schema(db_path: Path) -> None:
    con = connect_db(db_path)
    con.executescript(
        """
        CREATE TABLE IF NOT EXISTS Dim_Product (
            ProductKey     INTEGER PRIMARY KEY AUTOINCREMENT,
            ProductNr      INTEGER NOT NULL,
            ProductType    TEXT NOT NULL CHECK (ProductType IN ('Fiets','Accessoire')),
            Naam           TEXT,
            Merk           TEXT,
            Soort          TEXT,
            Type           TEXT,
            Kleur          TEXT,
            Standaardprijs REAL,
            Inkoopprijs    REAL,
            ValidFrom      TEXT NOT NULL,
            ValidTo        TEXT,
            IsCurrent      INTEGER NOT NULL DEFAULT 1 CHECK (IsCurrent IN (0,1))
        );

        CREATE TABLE IF NOT EXISTS Dim_Partner (
            PartnerKey     INTEGER PRIMARY KEY AUTOINCREMENT,
            PartnerNr      INTEGER NOT NULL,
            PartnerType    TEXT NOT NULL CHECK (PartnerType IN ('Fabrikant','Leverancier')),
            Naam           TEXT,
            Adres          TEXT,
            Plaats         TEXT,
            UNIQUE (PartnerType, PartnerNr)
        );

        CREATE TABLE IF NOT EXISTS Dim_Klant (
            KlantKey            INTEGER PRIMARY KEY AUTOINCREMENT,
            KlantNr             INTEGER NOT NULL,
            Naam                TEXT NOT NULL,
            Adres               TEXT,
            Woonplaats          TEXT,
            Geslacht            TEXT CHECK (Geslacht IN ('M','V','X') OR Geslacht IS NULL),
            Geboortedatum       TEXT,
            Leeftijd            INTEGER,
            Leeftijdscategorie  TEXT,
            ValidFrom           TEXT NOT NULL,
            ValidTo             TEXT,
            IsCurrent           INTEGER NOT NULL DEFAULT 1 CHECK (IsCurrent IN (0,1))
        );

        CREATE TABLE IF NOT EXISTS Dim_Monteur (
            MonteurKey     INTEGER PRIMARY KEY AUTOINCREMENT,
            MonteurNr      INTEGER NOT NULL UNIQUE,
            Naam           TEXT NOT NULL,
            Woonplaats     TEXT,
            Uurloon        REAL
        );

        CREATE TABLE IF NOT EXISTS Dim_Filiaal (
            FiliaalKey     INTEGER PRIMARY KEY AUTOINCREMENT,
            FiliaalNr      INTEGER NOT NULL UNIQUE,
            Naam           TEXT NOT NULL,
            Adres          TEXT,
            Provincie      TEXT
        );

        CREATE TABLE IF NOT EXISTS Dim_Datum (
            DatumKey       INTEGER PRIMARY KEY AUTOINCREMENT,
            Datum          TEXT NOT NULL UNIQUE,
            Dag            INTEGER NOT NULL,
            Maand          INTEGER NOT NULL,
            Jaar           INTEGER NOT NULL,
            Kwartaal       INTEGER NOT NULL,
            Weekdag        TEXT NOT NULL,
            Seizoen        TEXT,
            IsWeekend      INTEGER NOT NULL CHECK (IsWeekend IN (0,1))
        );

        CREATE TABLE IF NOT EXISTS Dim_Tijd (
            TijdKey        INTEGER PRIMARY KEY AUTOINCREMENT,
            Tijd           TEXT NOT NULL UNIQUE,
            Uur            INTEGER NOT NULL,
            Minuut         INTEGER NOT NULL,
            Dagdeel        TEXT
        );

        CREATE TABLE IF NOT EXISTS Fact_Inkoop (
            InkoopKey      INTEGER PRIMARY KEY AUTOINCREMENT,
            InkoopNr       INTEGER NOT NULL UNIQUE,
            ProductKey     INTEGER NOT NULL,
            PartnerKey     INTEGER NOT NULL,
            DatumKey       INTEGER NOT NULL,
            Aantal         INTEGER NOT NULL,
            Inkoopprijs    REAL NOT NULL,
            Inkoopbedrag   REAL NOT NULL,
            KortingBedrag  REAL,
            FOREIGN KEY (ProductKey) REFERENCES Dim_Product(ProductKey),
            FOREIGN KEY (PartnerKey) REFERENCES Dim_Partner(PartnerKey),
            FOREIGN KEY (DatumKey)   REFERENCES Dim_Datum(DatumKey)
        );

        CREATE TABLE IF NOT EXISTS Fact_Verkoop (
            VerkoopKey     INTEGER PRIMARY KEY AUTOINCREMENT,
            VerkoopNr      INTEGER NOT NULL UNIQUE,
            ProductKey     INTEGER NOT NULL,
            KlantKey       INTEGER NOT NULL,
            MonteurKey     INTEGER NOT NULL,
            FiliaalKey     INTEGER NOT NULL,
            DatumKey       INTEGER NOT NULL,
            Aantal         INTEGER NOT NULL,
            Verkoopprijs   REAL NOT NULL,
            Omzet          REAL NOT NULL,
            Inkoopbedrag   REAL,
            Brutowinst     REAL,
            FOREIGN KEY (ProductKey) REFERENCES Dim_Product(ProductKey),
            FOREIGN KEY (KlantKey)   REFERENCES Dim_Klant(KlantKey),
            FOREIGN KEY (MonteurKey) REFERENCES Dim_Monteur(MonteurKey),
            FOREIGN KEY (FiliaalKey) REFERENCES Dim_Filiaal(FiliaalKey),
            FOREIGN KEY (DatumKey)   REFERENCES Dim_Datum(DatumKey)
        );

        CREATE TABLE IF NOT EXISTS Fact_Onderhoud (
            OnderhoudKey       INTEGER PRIMARY KEY AUTOINCREMENT,
            OnderhoudNr        INTEGER NOT NULL UNIQUE,
            ProductKey         INTEGER NOT NULL,
            MonteurKey         INTEGER NOT NULL,
            FiliaalKey         INTEGER NOT NULL,
            DatumKey           INTEGER NOT NULL,
            StartTijdKey       INTEGER NOT NULL,
            EindTijdKey        INTEGER NOT NULL,
            AantalOnderhoud    INTEGER NOT NULL DEFAULT 1,
            OnderhoudsduurMin  INTEGER NOT NULL,
            Arbeidskosten      REAL NOT NULL,
            FOREIGN KEY (ProductKey)   REFERENCES Dim_Product(ProductKey),
            FOREIGN KEY (MonteurKey)   REFERENCES Dim_Monteur(MonteurKey),
            FOREIGN KEY (FiliaalKey)   REFERENCES Dim_Filiaal(FiliaalKey),
            FOREIGN KEY (DatumKey)     REFERENCES Dim_Datum(DatumKey),
            FOREIGN KEY (StartTijdKey) REFERENCES Dim_Tijd(TijdKey),
            FOREIGN KEY (EindTijdKey)  REFERENCES Dim_Tijd(TijdKey)
        );
        """
    )
    # Maak partiële unieke indexen zodat er maar één 'current' rij per natuurlijke sleutel is
    con.executescript(
        """
        CREATE UNIQUE INDEX IF NOT EXISTS UX_Dim_Product_Current
        ON Dim_Product (ProductType, ProductNr)
        WHERE IsCurrent = 1;

        CREATE UNIQUE INDEX IF NOT EXISTS UX_Dim_Klant_Current
        ON Dim_Klant (KlantNr)
        WHERE IsCurrent = 1;
        """
    )
    con.commit()
    con.close()

In [63]:
# Hulpfuncties voor datatransformatie
def normalize_time(time_str: str) -> str:
    return str(time_str).split(".")[0]

def minutes_between(start: str, end: str) -> int:
    fmt = "%H:%M:%S"
    start_dt = dt.datetime.strptime(normalize_time(start), fmt)
    end_dt = dt.datetime.strptime(normalize_time(end), fmt)
    return int((end_dt - start_dt).total_seconds() // 60)

def age_from_birthdate(date_str: str, ref_date: dt.date | None = None) -> int | None:
    if not date_str:
        return None
    if ref_date is None:
        ref_date = dt.date.today()
    birth = dt.date.fromisoformat(str(date_str)[:10])
    age = ref_date.year - birth.year - ((ref_date.month, ref_date.day) < (birth.month, birth.day))
    return age

def age_category(age: int | None) -> str | None:
    if age is None:
        return None
    if age < 18:
        return "<18"
    if age <= 30:
        return "18-30"
    if age <= 50:
        return "31-50"
    return "51+"

def season(month: int) -> str:
    return (
        "Winter" if month in (12,1,2)
        else "Lente" if month in (3,4,5)
        else "Zomer" if month in (6,7,8)
        else "Herfst"
    )

def dagdeel(hour: int) -> str:
    return "Ochtend" if hour < 12 else ("Middag" if hour < 18 else "Avond")

### Gegevens extraheren en transformeren vanuit het SDM

In [64]:
# Lees de SDM en bouw per tabel een DataFrame dat past bij het DWH‑schema
def extract_sdm_data(sdm_path: Path) -> dict[str, pd.DataFrame]:
    con = connect_db(sdm_path)
    def read(table: str) -> pd.DataFrame:
        return pd.read_sql_query(f"SELECT * FROM {table}", con)

    # Dim_Product: combineer fiets‑ en accessoiregegevens uit alle bronnen
    def make_product_df() -> pd.DataFrame:
        cols = ["ProductType","ProductNr","Naam","Merk","Soort","Type","Kleur","Standaardprijs","Inkoopprijs"]
        bike_sources = [
            read("Fiets_Inkoop_Fiets").assign(ProductType="Fiets"),
            read("Fietsverkoop_Fiets").assign(ProductType="Fiets"),
            read("Onderhoud_Fiets").assign(ProductType="Fiets"),
        ]
        bike = pd.concat(bike_sources, ignore_index=True).drop_duplicates(subset=["ProductType","fietsnr"])
        bike = bike.rename(columns={
            "fietsnr":"ProductNr","naam":"Naam","merk":"Merk","soort":"Soort",
            "type":"Type","kleur":"Kleur","standaardprijs":"Standaardprijs",
            "inkoopprijs":"Inkoopprijs"
        })
        acc_sources = [
            read("Accessoire_Inkoop_Accessoire").assign(ProductType="Accessoire"),
            read("Accessoireverkoop_Accessoire").assign(ProductType="Accessoire"),
        ]
        acc = pd.concat(acc_sources, ignore_index=True).drop_duplicates(subset=["ProductType","accessoirenr"])
        acc = acc.rename(columns={
            "accessoirenr":"ProductNr","naam":"Naam","soort":"Soort",
            "standaardprijs":"Standaardprijs","inkoopprijs":"Inkoopprijs"
        })
        # ontbrekende kolommen in accessoires vullen met None
        for df in [bike, acc]:
            for col in cols:
                if col not in df.columns:
                    df[col] = None
        return pd.concat([bike[cols], acc[cols]], ignore_index=True).drop_duplicates()

    # Dim_Partner: fabrikanten en leveranciers combineren
    def make_partner_df() -> pd.DataFrame:
        cols = ["PartnerType","PartnerNr","Naam","Adres","Plaats"]
        fabrikanten = read("Fiets_Inkoop_Fabrikant").assign(PartnerType="Fabrikant")
        fabrikanten = fabrikanten.rename(columns={
            "fabrikantnr":"PartnerNr","naam":"Naam","adres":"Adres","plaats":"Plaats"
        })
        leveranciers = read("Accessoire_Inkoop_Leverancier").assign(PartnerType="Leverancier")
        leveranciers = leveranciers.rename(columns={
            "leveranciernr":"PartnerNr","naam":"Naam","adres":"Adres","woonplaats":"Plaats"
        })
        return pd.concat([fabrikanten[cols], leveranciers[cols]], ignore_index=True).drop_duplicates()

    # Dim_Klant: klanten samenvoegen en leeftijd berekenen
    def make_klant_df() -> pd.DataFrame:
        cols = ["KlantNr","Naam","Adres","Woonplaats","Geslacht","Geboortedatum","Leeftijd","Leeftijdscategorie"]
        kv = read("Fietsverkoop_Klant").rename(columns={
            "klantnr":"KlantNr","naam":"Naam","adres":"Adres",
            "woonplaats":"Woonplaats","geslacht":"Geslacht","geboortedatum":"Geboortedatum"
        })
        ka = read("Accessoireverkoop_Klant").rename(columns={
            "klantnr":"KlantNr","naam":"Naam","adres":"Adres",
            "woonplaats":"Woonplaats","geslacht":"Geslacht","geboortedatum":"Geboortedatum"
        })
        klanten = pd.concat([kv, ka], ignore_index=True)
        klanten["Leeftijd"] = klanten["Geboortedatum"].apply(lambda d: age_from_birthdate(d))
        klanten["Leeftijdscategorie"] = klanten["Leeftijd"].apply(age_category)
        # verwijder dubbele klanten en behoud de laatste variant (zo blijft natuurlijke sleutel uniek)
        klanten = klanten.sort_values(by=["KlantNr"]).drop_duplicates(subset=["KlantNr"], keep="last")
        return klanten[cols]

    # Dim_Monteur: unieke monteurs uit alle bronnen
    def make_monteur_df() -> pd.DataFrame:
        cols = ["MonteurNr","Naam","Woonplaats","Uurloon"]
        bronnen = [read("Fietsverkoop_Monteur"), read("Accessoireverkoop_Monteur"), read("Onderhoud_Monteur")]
        monteurs = pd.concat(bronnen, ignore_index=True).drop_duplicates(subset=["monteurnr"])
        monteurs = monteurs.rename(columns={
            "monteurnr":"MonteurNr","naam":"Naam","woonplaats":"Woonplaats","uurloon":"Uurloon"
        })
        return monteurs[cols]

    # Dim_Filiaal: filialen uit verkoop en onderhoud
    def make_filiaal_df() -> pd.DataFrame:
        cols = ["FiliaalNr","Naam","Adres","Provincie"]
        bronnen = [read("Fietsverkoop_Filiaal"), read("Accessoireverkoop_Filiaal"), read("Onderhoud_Filiaal")]
        filialen = pd.concat(bronnen, ignore_index=True).drop_duplicates(subset=["filiaalnr"])
        filialen = filialen.rename(columns={
            "filiaalnr":"FiliaalNr","naam":"Naam","adres":"Adres","provincie":"Provincie"
        })
        return filialen[cols]

    # Dim_Datum: verzamel alle datums uit verkoop, onderhoud en inkoop
    def make_datum_df() -> pd.DataFrame:
        bike_dates = read("Fietsverkoop_Fiets_Verkoop")["datum"].dropna().astype(str)
        acc_dates  = read("Accessoireverkoop_Accessoire_Verkoop")["datum"].dropna().astype(str)
        maintenance = read("Onderhoud")["datum"].dropna().astype(str)
        fiets_inkoop = read("Fiets_Inkoop").assign(
            Datum=lambda df: df.apply(lambda r: f"{int(r['inkoopjaar']):04d}-{int(r['inkoopmaand']):02d}-01", axis=1)
        )["Datum"]
        acc_inkoop  = read("Accessoire_Inkoop").assign(
            Datum=lambda df: df.apply(lambda r: f"{int(r['inkoopjaar']):04d}-{int(r['inkoopmaand']):02d}-01", axis=1)
        )["Datum"]
        all_dates = pd.concat([bike_dates, acc_dates, maintenance, fiets_inkoop, acc_inkoop], ignore_index=True)
        all_dates = pd.to_datetime(all_dates.dropna()).drop_duplicates().sort_values()
        df = pd.DataFrame({"Datum": all_dates.dt.strftime("%Y-%m-%d")})
        dt_values = pd.to_datetime(df["Datum"])
        df["Dag"] = dt_values.dt.day
        df["Maand"] = dt_values.dt.month
        df["Jaar"] = dt_values.dt.year
        df["Kwartaal"] = dt_values.dt.quarter
        df["Weekdag"] = dt_values.dt.day_name()
        df["Seizoen"] = df["Maand"].apply(lambda m: season(int(m)))
        df["IsWeekend"] = dt_values.dt.dayofweek.isin([5,6]).astype(int)
        return df

    # Dim_Tijd: unieke start‑ en eindtijden omzetten naar hh:mm:ss
    def make_tijd_df() -> pd.DataFrame:
        tijden = pd.concat([
            read("Onderhoud")["starttijd"].apply(normalize_time),
            read("Onderhoud")["eindtijd"].apply(normalize_time)
        ], ignore_index=True).dropna().drop_duplicates().sort_values()
        df = pd.DataFrame({"Tijd": tijden})
        tijd_values = pd.to_datetime(df["Tijd"], format="%H:%M:%S")
        df["Uur"] = tijd_values.dt.hour
        df["Minuut"] = tijd_values.dt.minute
        df["Dagdeel"] = df["Uur"].apply(lambda h: dagdeel(int(h)))
        return df

    # Fact_Inkoop: fiets‑ en accessoire‑inkopen combineren en uniek nummer geven
    def make_fact_inkoop_df() -> pd.DataFrame:
        fi = read("Fiets_Inkoop").merge(read("Fiets_Inkoop_Fiets"), left_on="fiets", right_on="fietsnr")
        bike = pd.DataFrame({
            "InkoopNr": fi["inkoopnr"],
            "ProductType": "Fiets",
            "ProductNr": fi["fiets"],
            "PartnerType": "Fabrikant",
            "PartnerNr": fi["fabrikant"],
            "Datum": fi.apply(lambda r: f"{int(r['inkoopjaar']):04d}-{int(r['inkoopmaand']):02d}-01", axis=1),
            "Aantal": fi["aantal"],
            "Inkoopprijs": fi["inkoopprijs"],
            "Inkoopbedrag": (fi["aantal"] * fi["inkoopprijs"]).round(2),
            "KortingBedrag": ((fi["standaardprijs"] - fi["inkoopprijs"]) * fi["aantal"]).round(2),
        })
        ai = read("Accessoire_Inkoop").merge(read("Accessoire_Inkoop_Accessoire"), left_on="accessoire", right_on="accessoirenr")
        acc = pd.DataFrame({
            # voeg een grote offset toe zodat InkoopNr uniek blijft over alle bronnen
            "InkoopNr": ai["inkoopnr"] + 1000000,
            "ProductType": "Accessoire",
            "ProductNr": ai["accessoire"],
            "PartnerType": "Leverancier",
            "PartnerNr": ai["leverancier"],
            "Datum": ai.apply(lambda r: f"{int(r['inkoopjaar']):04d}-{int(r['inkoopmaand']):02d}-01", axis=1),
            "Aantal": ai["aantal"],
            "Inkoopprijs": ai["inkoopprijs"],
            "Inkoopbedrag": (ai["aantal"] * ai["inkoopprijs"]).round(2),
            "KortingBedrag": ((ai["standaardprijs"] - ai["inkoopprijs"]) * ai["aantal"]).round(2),
        })
        return pd.concat([bike, acc], ignore_index=True)

    # Fact_Verkoop: fiets‑ en accessoireverkopen combineren en uniek nummer geven
    def make_fact_verkoop_df() -> pd.DataFrame:
        bikev = read("Fietsverkoop_Fiets_Verkoop").merge(
            read("Fietsverkoop_Fiets"), left_on="fiets", right_on="fietsnr"
        ).merge(
            read("Fietsverkoop_Monteur")[["monteurnr","filiaal"]], left_on="monteur", right_on="monteurnr"
        )
        bike_fact = pd.DataFrame({
            "VerkoopNr": bikev["fiets_verkoopnr"],
            "ProductType": "Fiets",
            "ProductNr": bikev["fiets"],
            "KlantNr": bikev["klant"],
            "MonteurNr": bikev["monteur"],
            "FiliaalNr": bikev["filiaal"],
            "Datum": bikev["datum"].astype(str),
            "Aantal": bikev["aantal"],
            "Verkoopprijs": bikev["verkoopprijs"],
            "Omzet": (bikev["aantal"] * bikev["verkoopprijs"]).round(2),
            "Inkoopbedrag": (bikev["aantal"] * bikev["inkoopprijs"]).round(2),
            "Brutowinst": ((bikev["aantal"] * bikev["verkoopprijs"]) - (bikev["aantal"] * bikev["inkoopprijs"])).round(2),
        })
        accv = read("Accessoireverkoop_Accessoire_Verkoop").merge(
            read("Accessoireverkoop_Accessoire"), left_on="accessoire", right_on="accessoirenr"
        ).merge(
            read("Accessoireverkoop_Monteur")[["monteurnr","filiaal"]], left_on="monteur", right_on="monteurnr"
        )
        acc_fact = pd.DataFrame({
            "VerkoopNr": accv["accessoire_verkoopnr"] + 1000000,
            "ProductType": "Accessoire",
            "ProductNr": accv["accessoire"],
            "KlantNr": accv["klant"],
            "MonteurNr": accv["monteur"],
            "FiliaalNr": accv["filiaal"],
            "Datum": accv["datum"].astype(str),
            "Aantal": accv["aantal"],
            "Verkoopprijs": accv["verkoopprijs"],
            "Omzet": (accv["aantal"] * accv["verkoopprijs"]).round(2),
            "Inkoopbedrag": (accv["aantal"] * accv["inkoopprijs"]).round(2),
            "Brutowinst": ((accv["aantal"] * accv["verkoopprijs"]) - (accv["aantal"] * accv["inkoopprijs"])).round(2),
        })
        return pd.concat([bike_fact, acc_fact], ignore_index=True)

    # Fact_Onderhoud: onderhoudsfeiten opbouwen
    def make_fact_onderhoud_df() -> pd.DataFrame:
        ond = read("Onderhoud").merge(
            read("Onderhoud_Monteur")[["monteurnr","filiaal","uurloon"]],
            left_on="monteur", right_on="monteurnr"
        )
        return pd.DataFrame({
            "OnderhoudNr": ond["onderhoudnr"],
            "ProductType": "Fiets",
            "ProductNr": ond["fiets"],
            "MonteurNr": ond["monteur"],
            "FiliaalNr": ond["filiaal"],
            "Datum": ond["datum"].astype(str).str[:10],
            "StartTijd": ond["starttijd"].apply(normalize_time),
            "EindTijd": ond["eindtijd"].apply(normalize_time),
            "AantalOnderhoud": 1,
            "OnderhoudsduurMin": ond.apply(lambda r: minutes_between(r["starttijd"], r["eindtijd"]), axis=1),
            "Arbeidskosten": ond.apply(
                lambda r: round((minutes_between(r["starttijd"], r["eindtijd"]) / 60.0) * float(r["uurloon"]), 2),
                axis=1
            ),
        })

    result = {
        "Dim_Product": make_product_df(),
        "Dim_Partner": make_partner_df(),
        "Dim_Klant": make_klant_df(),
        "Dim_Monteur": make_monteur_df(),
        "Dim_Filiaal": make_filiaal_df(),
        "Dim_Datum": make_datum_df(),
        "Dim_Tijd": make_tijd_df(),
        "Fact_Inkoop": make_fact_inkoop_df(),
        "Fact_Verkoop": make_fact_verkoop_df(),
        "Fact_Onderhoud": make_fact_onderhoud_df(),
    }
    con.close()
    return result

### SCD‑detectie en updatefuncties

In [65]:
# Bepaal welke bronrijen nieuw, gewijzigd of ongewijzigd zijn t.o.v. de bestaande DWH‑rijen
def detect_scd_changes(source: pd.DataFrame, target_current: pd.DataFrame,
                       business_keys: List[str], compare_cols: List[str]) -> Dict[str, pd.DataFrame]:
    if target_current.empty:
        return {
            "new": source.copy().reset_index(drop=True),
            "changed": pd.DataFrame(columns=source.columns),
            "unchanged": pd.DataFrame(columns=source.columns),
        }
    merged = source.merge(target_current, on=business_keys, how="left",
                          suffixes=("_src", "_tgt"), indicator=True)
    new_mask = merged["_merge"] == "left_only"
    both_mask = merged["_merge"] == "both"
    # bepaal verschil per kolom: leeg/NaN wordt gelijk behandeld
    if compare_cols:
        diffs = []
        for col in compare_cols:
            src_col = merged[f"{col}_src"]
            tgt_col = merged[f"{col}_tgt"]
            diff = ~((src_col == tgt_col) | (src_col.isna() & tgt_col.isna()))
            diffs.append(diff)
        diff_df = pd.concat(diffs, axis=1)
        changed_mask = both_mask & diff_df.any(axis=1)
    else:
        changed_mask = pd.Series(False, index=merged.index)
    unchanged_mask = both_mask & ~changed_mask
    def extract(mask: pd.Series) -> pd.DataFrame:
        data = {}
        for col in source.columns:
            if col in business_keys:
                data[col] = merged.loc[mask, col]
            else:
                data[col] = merged.loc[mask, f"{col}_src"]
        return pd.DataFrame(data).reset_index(drop=True)
    return {
        "new": extract(new_mask),
        "changed": extract(changed_mask),
        "unchanged": extract(unchanged_mask),
    }

In [66]:
# SCD Type 1: nieuwe rijen invoegen en gewijzigde attributen overschrijven
def upsert_scd1(con: sqlite3.Connection, table: str, source_df: pd.DataFrame,
                business_keys: List[str], compare_cols: List[str]) -> Dict[str, int]:
    cols_to_select = business_keys + compare_cols
    target_df = pd.read_sql_query(f"SELECT {', '.join(cols_to_select)} FROM {table}", con)
    deltas = detect_scd_changes(source_df, target_df, business_keys, compare_cols)
    cur = con.cursor()
    insert_sql = f"INSERT INTO {table} ({', '.join(source_df.columns)}) VALUES ({', '.join(['?']*len(source_df.columns))})"
    if compare_cols:
        set_clause = ", ".join([f"{col}=?" for col in compare_cols])
        where_clause = " AND ".join([f"{key}=?" for key in business_keys])
        update_sql = f"UPDATE {table} SET {set_clause} WHERE {where_clause}"
    else:
        update_sql = None
    # insert nieuwe rijen
    for _, row in deltas["new"].iterrows():
        cur.execute(insert_sql, [row[col] if pd.notna(row[col]) else None for col in source_df.columns])
    # update gewijzigde rijen
    for _, row in deltas["changed"].iterrows():
        if update_sql:
            values = [row[col] if pd.notna(row[col]) else None for col in compare_cols]
            values += [row[key] for key in business_keys]
            cur.execute(update_sql, values)
    con.commit()
    return {k: len(v) for k,v in deltas.items()}

In [67]:
# SCD Type 2: nieuwe rijen invoegen, bestaande sluiten met ValidTo en IsCurrent
def upsert_scd2(con: sqlite3.Connection, table: str, source_df: pd.DataFrame,
                business_keys: List[str], compare_cols: List[str], load_ts: str) -> Dict[str,int]:
    cols_to_select = business_keys + compare_cols + ["ValidFrom","ValidTo","IsCurrent"]
    target_df = pd.read_sql_query(f"SELECT {', '.join(cols_to_select)} FROM {table} WHERE IsCurrent = 1", con)
    current_df = target_df[business_keys + compare_cols]
    deltas = detect_scd_changes(source_df, current_df, business_keys, compare_cols)
    cur = con.cursor()
    insert_cols = list(source_df.columns) + ["ValidFrom","ValidTo","IsCurrent"]
    insert_sql  = f"INSERT INTO {table} ({', '.join(insert_cols)}) VALUES ({', '.join(['?']*len(insert_cols))})"
    close_sql   = f"UPDATE {table} SET ValidTo = ?, IsCurrent = 0 WHERE " + " AND ".join([f"{k}=?" for k in business_keys]) + " AND IsCurrent = 1"
    for cat in ["new","changed"]:
        for _, row in deltas[cat].iterrows():
            if cat == "changed":
                cur.execute(close_sql, [load_ts] + [row[key] for key in business_keys])
            values = [row[col] if pd.notna(row[col]) else None for col in source_df.columns]
            cur.execute(insert_sql, values + [load_ts, None, 1])
    con.commit()
    return {k: len(v) for k,v in deltas.items()}

In [68]:
# Haal mapping van natuurlijke sleutels naar surrogaat­sleutels op
def get_surrogate_mapping(con: sqlite3.Connection, table: str,
                           natural_keys: List[str], surrogate: str,
                           current_only: bool = False) -> Dict[tuple, int]:
    sql = f"SELECT {', '.join(natural_keys)}, {surrogate} FROM {table}"
    if current_only:
        sql += " WHERE IsCurrent = 1"
    df = pd.read_sql_query(sql, con)
    mapping = {}
    for _, row in df.iterrows():
        key = tuple(row[k] for k in natural_keys)
        mapping[key] = row[surrogate]
    return mapping

In [69]:
# Voeg een surrogaatkey toe aan een DataFrame met behulp van een mapping
def add_lookup(df: pd.DataFrame, new_col: str, mapping: Dict[tuple,int], key_cols: List[str]) -> pd.DataFrame:
    def map_key(row):
        key = tuple(row[col] for col in key_cols)
        return mapping.get(key)
    df[new_col] = df.apply(map_key, axis=1)
    return df

In [70]:
# Insert alleen nieuwe fact‑rijen (geen updates)
def insert_new_facts(con: sqlite3.Connection, table: str, source_df: pd.DataFrame,
                     business_keys: List[str]) -> Dict[str,int]:
    existing = pd.read_sql_query(f"SELECT {', '.join(business_keys)} FROM {table}", con) if con.execute(
        f"SELECT count(name) FROM sqlite_master WHERE type='table' AND name='{table}'").fetchone()[0] else pd.DataFrame(columns=business_keys)
    merged = source_df.merge(existing, on=business_keys, how="left", indicator=True)
    new_rows = merged[merged["_merge"] == "left_only"][source_df.columns]
    cur = con.cursor()
    if not new_rows.empty:
        insert_sql = f"INSERT INTO {table} ({', '.join(source_df.columns)}) VALUES ({', '.join(['?']*len(source_df.columns))})"
        for _, row in new_rows.iterrows():
            cur.execute(insert_sql, [row[col] if pd.notna(row[col]) else None for col in source_df.columns])
        con.commit()
    return {"new": len(new_rows), "existing": len(source_df) - len(new_rows)}

### ETL uitvoeren (dimension load, fact load en overzicht)

In [71]:
def run_etl(sdm_path: Path, dws_path: Path) -> pd.DataFrame:
    # Stap 1: maak schema aan
    prepare_target_schema(dws_path)
    # Stap 2: extract
    src = extract_sdm_data(sdm_path)
    con = connect_db(dws_path)
    results = {}
    load_ts = dt.datetime.now().replace(microsecond=0).isoformat()
    # Configureer per dimensie SCD‑type en sleutelkolommen
    dim_cfg = {
        "Dim_Datum":  (1, ["Datum"], ["Dag","Maand","Jaar","Kwartaal","Weekdag","Seizoen","IsWeekend"]),
        "Dim_Tijd":   (1, ["Tijd"], ["Uur","Minuut","Dagdeel"]),
        "Dim_Partner":(1, ["PartnerType","PartnerNr"], ["Naam","Adres","Plaats"]),
        "Dim_Filiaal":(1, ["FiliaalNr"], ["Naam","Adres","Provincie"]),
        "Dim_Monteur":(1, ["MonteurNr"], ["Naam","Woonplaats","Uurloon"]),
        "Dim_Product":(2, ["ProductType","ProductNr"], ["Naam","Merk","Soort","Type","Kleur","Standaardprijs","Inkoopprijs"]),
        "Dim_Klant":  (2, ["KlantNr"], ["Naam","Adres","Woonplaats","Geslacht","Geboortedatum","Leeftijd","Leeftijdscategorie"]),
    }
    # Stap 3: laad dimensies
    for table, (scd_type, keys, attrs) in dim_cfg.items():
        df = src[table].copy()
        if scd_type == 1:
            res = upsert_scd1(con, table, df, keys, attrs)
        else:
            res = upsert_scd2(con, table, df, keys, attrs, load_ts)
        results[table] = res
    # Stap 4: maak mappings
    maps = {
        "Product": get_surrogate_mapping(con, "Dim_Product", ["ProductType","ProductNr"], "ProductKey", current_only=True),
        "Partner": get_surrogate_mapping(con, "Dim_Partner", ["PartnerType","PartnerNr"], "PartnerKey"),
        "Klant":   get_surrogate_mapping(con, "Dim_Klant", ["KlantNr"], "KlantKey", current_only=True),
        "Monteur": get_surrogate_mapping(con, "Dim_Monteur", ["MonteurNr"], "MonteurKey"),
        "Filiaal": get_surrogate_mapping(con, "Dim_Filiaal", ["FiliaalNr"], "FiliaalKey"),
        "Datum":   get_surrogate_mapping(con, "Dim_Datum", ["Datum"], "DatumKey"),
        "Tijd":    get_surrogate_mapping(con, "Dim_Tijd", ["Tijd"], "TijdKey"),
    }
    # Stap 5: laad fact_tabellen
    fact_inkoop = src["Fact_Inkoop"].copy()
    fact_inkoop = add_lookup(fact_inkoop, "ProductKey", maps["Product"], ["ProductType","ProductNr"])
    fact_inkoop = add_lookup(fact_inkoop, "PartnerKey", maps["Partner"], ["PartnerType","PartnerNr"])
    fact_inkoop["DatumKey"] = fact_inkoop["Datum"].map(lambda d: maps["Datum"].get((d,), None))
    fact_inkoop = fact_inkoop[["InkoopNr","ProductKey","PartnerKey","DatumKey","Aantal","Inkoopprijs","Inkoopbedrag","KortingBedrag"]]
    results["Fact_Inkoop"] = insert_new_facts(con, "Fact_Inkoop", fact_inkoop, ["InkoopNr"])

    fact_verkoop = src["Fact_Verkoop"].copy()
    fact_verkoop = add_lookup(fact_verkoop, "ProductKey", maps["Product"], ["ProductType","ProductNr"])
    fact_verkoop = add_lookup(fact_verkoop, "KlantKey", maps["Klant"], ["KlantNr"])
    fact_verkoop = add_lookup(fact_verkoop, "MonteurKey", maps["Monteur"], ["MonteurNr"])
    fact_verkoop = add_lookup(fact_verkoop, "FiliaalKey", maps["Filiaal"], ["FiliaalNr"])
    fact_verkoop["DatumKey"] = fact_verkoop["Datum"].map(lambda d: maps["Datum"].get((d,), None))
    fact_verkoop = fact_verkoop[["VerkoopNr","ProductKey","KlantKey","MonteurKey","FiliaalKey","DatumKey","Aantal","Verkoopprijs","Omzet","Inkoopbedrag","Brutowinst"]]
    results["Fact_Verkoop"] = insert_new_facts(con, "Fact_Verkoop", fact_verkoop, ["VerkoopNr"])

    fact_onderhoud = src["Fact_Onderhoud"].copy()
    fact_onderhoud = add_lookup(fact_onderhoud, "ProductKey", maps["Product"], ["ProductType","ProductNr"])
    fact_onderhoud = add_lookup(fact_onderhoud, "MonteurKey", maps["Monteur"], ["MonteurNr"])
    fact_onderhoud = add_lookup(fact_onderhoud, "FiliaalKey", maps["Filiaal"], ["FiliaalNr"])
    fact_onderhoud["DatumKey"]      = fact_onderhoud["Datum"].map(lambda d: maps["Datum"].get((d,), None))
    fact_onderhoud["StartTijdKey"] = fact_onderhoud["StartTijd"].map(lambda t: maps["Tijd"].get((t,), None))
    fact_onderhoud["EindTijdKey"]  = fact_onderhoud["EindTijd"].map(lambda t: maps["Tijd"].get((t,), None))
    fact_onderhoud = fact_onderhoud[["OnderhoudNr","ProductKey","MonteurKey","FiliaalKey","DatumKey","StartTijdKey","EindTijdKey","AantalOnderhoud","OnderhoudsduurMin","Arbeidskosten"]]
    results["Fact_Onderhoud"] = insert_new_facts(con, "Fact_Onderhoud", fact_onderhoud, ["OnderhoudNr"])
    con.close()
    # Resultaat samenvatten als DataFrame voor overzicht
    rows = []
    for table, res in results.items():
        for op, count in res.items():
            rows.append({"Tabel": table, "Soort": op, "Aantal": count})
    return pd.DataFrame(rows)

In [72]:
# Kies paden naar de bron‑ en doel‑databases
sdm_path = Path("database/SDM.db")
dws_path = Path("database/DWS.db")
# Voer de ETL uit en toon een overzicht
overzicht = run_etl(sdm_path, dws_path)
overview_sorted = overzicht.sort_values(["Tabel","Soort"]).reset_index(drop=True)
display(overview_sorted)

,Tabel,Soort,Aantal
0,Dim_Datum,changed,0
1,Dim_Datum,new,0
2,Dim_Datum,unchanged,201
3,Dim_Filiaal,changed,0
4,Dim_Filiaal,new,0
5,Dim_Filiaal,unchanged,5
6,Dim_Klant,changed,0
7,Dim_Klant,new,0
8,Dim_Klant,unchanged,25
9,Dim_Monteur,changed,0
